# Declarative Fine-Tuning with **Axolotl** — Continued Pre-training (CPT) / Domain Adaptation
### Tools & Frameworks  ·  Colab T4 (16 GB) ready

> The **CPT notebook** in `advanced-fine-tuning-paradigms/` builds domain adaptation by hand: quantise, attach LoRA, tokenise, pack, schedule, evaluate — roughly 350 lines of Python, each one a decision *and* a liability. Every one of those lines is a thing that can drift between your laptop, your teammate's box, and the 16-GPU cluster you actually train on.
> This notebook runs **the same algorithm** with the Python deleted. One `config.yml`, one CLI call. The interesting content is not "Axolotl has a YAML file" — it is **which decisions survive the translation, which ones the schema now makes for you, and the four places where the declarative abstraction has a sharp edge you must know about.**
>
> Read it side-by-side with `cpt.ipynb`: same base model, same corpora, same recipe, same before/after perplexity pair.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

**The payload: CPT.**

- **CPT** — in the literature **DAPT, Domain-Adaptive Pretraining** (*Gururangan et al., 2020*) — is the **warm-started continuation of the causal-LM objective** on a domain corpus, initialised from converged base weights:
  $$\mathcal{L}_{\text{CPT}}(\theta) = -\,\mathbb{E}_{x \sim \mathcal{D}_{\text{domain}}}\Big[\sum_{t=1}^{|x|} \log \pi_\theta(x_t \mid x_{<t})\Big]$$
- The loss is **un-masked over every token**: no prompt/completion split, no `-100`, no chat template. SFT differentiates a *conditional* `p(y|x)` over ~20–40 % of forwarded tokens; CPT differentiates the *unconditional prior* `p(x)` over **100 %**.
- The canonical modern recipe (*Ibrahim et al., 2024*) is three ingredients: **re-warm** the LR, **re-decay** it on a fresh cosine, and **replay 1–5 %** of general-distribution text against catastrophic forgetting.

**The vehicle: declarative training configuration.**

- **Declarative config** — the run is described as *data* (a YAML document), never as *control flow*. The training program is fixed and general; your experiment is the argument to it. This is the same inversion as Terraform-vs-`aws` shell scripts or Kubernetes manifests-vs-`docker run`.
- **Schema-validated config** — Axolotl parses the YAML into a **Pydantic model** (`AxolotlInputConfig` in `axolotl.utils.schemas.config`) with ~600 typed fields and dozens of cross-field `@model_validator`s. Illegal *combinations* — not just illegal values — are rejected before a single weight is loaded. `axolotl config-schema --field <name>` dumps the contract for your pinned version.
- **Prompt strategy registry** — `type:` in a dataset entry names a *registered tokenisation strategy*, not a format string. For CPT the two that matter are:
  - **`type: completion`** — raw text from `field:`, labels **unmasked**, and (verified in `prompt_strategies/completion.py`) each document is **sliced into `sequence_len`-token rows automatically**, up to `sequence_len × 64` per document. This is the finite/map-style path.
  - **`type: pretrain`** — the **streaming** path (`pretraining_dataset:`), which tokenises on the fly and, with `pretraining_sample_concatenation: true`, does classic **concat-and-chunk** across document boundaries with `eos` separators.
- **Multipack** — Axolotl's Best-Fit-Decreasing packer (`sample_packing: true`), which fills each row with several whole documents and isolates them with a **block-diagonal attention mask** (`pretrain_multipack_attn`). It rides on FlashAttention-2's `varlen` path — the reason it is Ampere-and-up only, and the reason this notebook turns it off on a T4.
- **Pre-tokenisation cache** — `dataset_prepared_path` stores the tokenised Arrow dataset keyed by a hash of the dataset config. `axolotl preprocess` fills it on **one** process; every rank in a distributed job then memory-maps the identical bytes instead of racing to tokenise.
- **Launcher indirection** — `axolotl train` is a thin wrapper that `exec`s `accelerate launch -m axolotl.cli.train` (or `torchrun`, or `python`, via `--launcher`). *Topology* — how many machines, which rank, which ZeRO stage — is supplied by the launcher and by `deepspeed:` / `fsdp_config:` keys. **Not one line of your training code changes between 1 GPU and 16.**

### One-sentence definition of the mechanics

> **Axolotl compiles a single version-controlled YAML document into a fully-configured `Trainer` — model loading and quantisation, dataset resolution, prompt tokenisation, packing, LR schedule, evaluation and checkpointing — and hands it to `accelerate`, so that executing continued pre-training on one T4 or on sixteen H100s across two nodes is the same file with a different launcher.**

### The exact engineering problem it solves

CPT is the paradigm where this matters most, for a reason that is arithmetic rather than aesthetic:

- **CPT runs are long, which makes irreproducibility expensive.** SFT on 50 k pairs is a 40-minute run; if the script drifted, you rerun it. Real CPT is 10⁹–10¹¹ tokens — **days on a multi-node cluster**. "Which learning rate did the run that worked use?" must be answerable from `git log`, not from a Slack scrollback.
- **CPT is the paradigm most likely to be multi-node,** because it is compute-bound (`FLOPs ≈ 6·N·D`) rather than data-bound. Multi-node is exactly where hand-rolled scripts fail: `dist.init_process_group` ordering, rank-0-only dataset preparation, `NCCL_SOCKET_IFNAME`, gradient-checkpointing plus ZeRO-3 interactions, resharded checkpoint loading. Every one of those is boilerplate — identical across projects, subtly wrong in each.
- **The tokenisation race.** A 16-rank job that each independently tokenises 30 GB of raw text wastes 15/16 of that CPU work and can deadlock on the Arrow cache lock. `axolotl preprocess` makes this a **separate, cacheable, CPU-only build step** — and one you can run on a cheap box before ever booking a GPU.
- **Config errors are the most expensive class of bug in training.** `sample_packing` without FlashAttention, `pretraining_dataset` with `val_set_size > 0`, a mistyped `field:` — in a hand-rolled script each of these is discovered by an exception 40 minutes in, or worse, by a loss curve that is quietly wrong. A Pydantic validator rejects them in **200 ms**.
- **Hyperparameter provenance.** The config file *is* the experiment record. Diffing two runs is `diff a.yml b.yml`, not archaeology across two Python files.

> What it does **not** solve: **data curation.** Everything upstream of "a column of clean text" — dedup, licence review, PII, mixture ratios — stays yours. Section 3 is honest about that boundary and puts it in one clearly-labelled cell.

---

### The Human Element — Hugging Face datasets for CPT, and their YAML contract

CPT's data contract is *one column of unlabelled text*. What differs between corpora is **which column, how long the rows are, and whether the corpus fits on disk** — and each of those maps to a specific set of YAML keys.

| HF path | What it is | Why it is shaped this way, and how it is declared |
|---|---|---|
| **`epfl-llm/guidelines`**<br>`train`, 37,970 docs, ~865 MB | The **clinical-practice-guideline corpus behind Meditron-7B/70B** — full-length documents from 16 medical sources in a single `clean_text` column. | The reference shape of a CPT corpus: **long, unstructured prose; no prompt, no completion, no schema.** Length is a *feature* — long documents pack into 1–8 k windows with near-zero padding and carry the domain's discourse structure (guideline → evidence grade → recommendation). Declared as `type: completion` + `field: clean_text`; Axolotl slices each document into `sequence_len` rows for you. ⚠️ Some rows are `None`/stubs where a source was dropped for licensing — the tokeniser will raise on those, which is exactly why Step 1 filters before writing. |
| **`MedRAG/textbooks`**<br>`train`, 125,847 rows | 18 medical textbooks pre-chunked into paragraph-sized `content` rows. | The **opposite extreme of the same contract** — same "one text column, no labels", but rows are short. It is the clearest case for `sample_packing: true`: fed one row per sequence into a 1024-token window, most of every row would be padding. Declared identically with `field: content`. Better when you want the domain's *definitional* register rather than its *procedural* one. |
| **`HuggingFaceFW/fineweb-edu`**<br>config **`sample-10BT`**, `text` | High-quality general web/educational text, streamable at any scale. | The **replay** half of the mixture, and it is not optional — the anti-forgetting recipe needs a few percent of the *original* pretraining distribution, and since no open base model publishes its exact mix, a high-quality general corpus is the standard proxy. Structured identically to the domain corpus **precisely so a second `datasets:` entry is the entire implementation of replay** — Axolotl concatenates and shuffles the entries (`shuffle_merged_datasets: true`), so the mixture ratio is a line of YAML. At 10 B tokens it is also the canonical `pretraining_dataset:` + `type: pretrain` streaming demo. |
| **`allenai/peS2o`** *(scale-up)* | ~40 M full-text scientific papers — the canonical large-scale science CPT corpus. | Same single-text-column contract at the 10¹⁰-token scale you would actually book a cluster for; this is where `pretraining_dataset:` streaming stops being a demo. ⚠️ Ships a loading script, so it needs `datasets<4.0` with `trust_remote_code: true` or a direct parquet fetch. |

**Why they are all one unlabelled column:** CPT's objective has no notion of input vs. target — the target *is* the shifted input. Any extra structure (roles, pairs, scores) is unusable by the loss and gets dropped. In a declarative pipeline this is a gift: the entire data specification collapses to `path` / `ds_type` / `type` / `field`, which is why CPT is the cleanest possible demonstration of config-driven training.

> This notebook trains on **`epfl-llm/guidelines`** (domain) mixed with **`HuggingFaceFW/fineweb-edu` / `sample-10BT`** (replay), starting from **`Qwen/Qwen2.5-0.5B` — the base, deliberately *not* `-Instruct`**, because CPT precedes instruction tuning.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **A training run has a type signature, and YAML lets you check it.** The set of *valid* CPT configurations is small and full of non-obvious cross-field constraints: 4-bit weights require an adapter (you cannot backprop into NF4); `sample_packing` requires a varlen attention kernel; `pretraining_dataset` is a stream and therefore has no length, so it requires `max_steps` and forbids `val_set_size`. In imperative code these are invariants living in your head. In Axolotl they are `@model_validator`s, and the failure moves from *minute 40 of an 8-GPU job* to *200 ms on your laptop*. This is the whole argument, and it is a type-systems argument.
- **Rank-invariance is the point of the abstraction.** The mathematical object being optimised — `(1-α)·L_domain + α·L_general` under a re-warmed cosine — is **independent of the parallelism used to compute it**. An imperative script fuses the two: `dist.init_process_group`, `DistributedSampler`, rank-guarded downloads, `local_rank` device placement, ZeRO-3 parameter gathering before `save_pretrained`. Axolotl keeps them orthogonal: the YAML is the objective, the launcher is the topology. `accelerate launch --num_machines 2 --machine_rank $RANK` is the *only* delta between a single T4 and a two-node cluster.
- **`preprocess` exists because tokenisation is not training.** Tokenising a CPT corpus is a CPU-bound, embarrassingly-parallel, perfectly-cacheable map over text. Doing it inside the training job burns GPU-hours at idle and multiplies the work by the world size. Splitting it out (`axolotl preprocess` → `dataset_prepared_path`) turns it into a **build artifact**: hash-keyed, computed once on a cheap box, memory-mapped read-only by every rank. On a real corpus this is the difference between 20 minutes of 16 idle H100s and zero.
- **Why the LR must be re-warmed, not resumed — and why that is a YAML line here.** A released base checkpoint is the *endpoint* of a decayed schedule and ships **without optimiser moments**. Too little LR and updates cannot leave the general-text basin (you buy style, not knowledge); too much and the first high-variance batches poison the moment estimates and forgetting turns catastrophic. `warmup_ratio: 0.05` + `lr_scheduler: cosine` + `cosine_min_lr_ratio` *is* the Ibrahim et al. recipe, stated once, in the file you commit.
- **Why replay works, mathematically, and why it is one more list item.** Mixing a fraction `α` of general data makes the true objective `(1-α)·L_domain + α·L_general`, keeping a gradient component pointed at the original distribution on every step; empirically **α ≈ 0.01–0.05** recovers most of full replay's retention for almost nothing. Because both corpora satisfy the same one-column contract, replay in Axolotl is **a second entry under `datasets:`** — the mixture ratio becomes a reviewable diff instead of a `concatenate_datasets` call buried 200 lines into a script.

#### The four sharp edges of this abstraction *(verified against `axolotl-ai-cloud/axolotl@main`, not the docs)*

Declarative does not mean magic. These are the places where the YAML quietly means something other than what it reads like — all four are load-bearing for CPT specifically:

1. **`pretraining_dataset:` is a list, but only element `[0]` is ever read.** `_extract_pretraining_config()` in `axolotl/utils/data/sft.py` takes `cfg.pretraining_dataset[0]` and discards the rest. **Adding your replay corpus as a second entry silently trains on zero replay.** On the streaming path, the mixture must be pre-mixed into one source; on the finite `datasets:` path (used below) multiple entries *are* merged and shuffled, which is why this notebook uses it.
2. **Concat-and-chunk is opt-in.** `pretraining_sample_concatenation` defaults to `null`; `encode_streaming(..., concatenate=cfg.pretraining_sample_concatenation is True)`. Leave it unset and every short document becomes its own padded row — a corpus of 400-token web pages in a 1024-token window runs at ~40 % utilisation, i.e. you pay a **2.5× cloud bill** for the same model.
3. **The streaming encoder truncates before it concatenates.** `encode_streaming` calls the tokeniser with `max_length=sequence_len - 2`, so on the `type: pretrain` path a 40 k-token guideline contributes only its first ~1 k tokens. The finite `type: completion` path does *not* have this behaviour — it slices documents into as many `sequence_len` rows as needed (up to `sequence_len × 64`). **The same corpus yields different training data depending on which key you used**, and nothing warns you.
4. **`sample_packing` is a hardware claim, not a data claim.** Multipack isolates packed documents via FlashAttention-2's `varlen` path, which needs `sm_80`+. On a T4 (`sm_75`) it must be off. There is also a live warning for `sample_packing` + `attn_implementation: sdpa` + `bf16` producing **0.0 loss** on anything below H100. Newer builds add `sdpa_varlen` (torch ≥ 2.10) as a flash-free alternative.

#### VRAM & Compute Impact

**Axolotl changes none of the training mathematics** — it is `transformers.Trainer` underneath, so the CPT cost model is unchanged:

- **`FLOPs ≈ 6·N·D`** (params × tokens, fwd+bwd). **LoRA/QLoRA does not reduce this** — frozen NF4 weights are still multiplied twice per step. PEFT buys optimiser/gradient *memory*, never CPT's dominant cost.
- CPT is **compute-bound**; SFT is **data-bound**. That single reframe drives every hardware key below.

| Component | Full-FT `AdamW` fp32 | **This config: QLoRA r=64, T4** |
|---|---|---|
| Base weights (0.5 B) | 2.0 GB fp16 | **0.40 GB** (`load_in_4bit` + `bnb_4bit_use_double_quant`) |
| Trainable params | 494 M | **~35 M** (`lora_target_linear: true` → all 7 projections × 24 layers) |
| Grads + optimiser state | ~5.9 GB | **~0.35 GB** (`optimizer: paged_adamw_8bit`) |
| Activations @ 1024×2, checkpointed | ~1.5 GB | **~1.5 GB** *(unchanged — same FLOPs)* |
| **Peak** | **~9–10 GB** | **~2.5–4 GB** |

What the *declarative layer* actually changes, in memory-and-compute terms:

- **Token utilisation is a config field, not a code path.** `sample_packing: true` moves a short-row corpus from ~40 % to ~99 % utilisation — at CPT's token counts that is not an optimisation, it is a **2.5× smaller bill**. `pad_to_sequence_len: true` keeps buffer shapes static, which reduces allocator fragmentation and the OOM-at-step-900 class of failure.
- **`gradient_checkpointing: true` + `use_reentrant: false`** — the standard CPT trade: ~30–40 % slower per step, activation memory from `O(layers)` to `O(√layers)`. `use_reentrant: false` is the non-optional half on any distributed run; the reentrant autograd path breaks with ZeRO-3 and with frozen-parameter graphs like LoRA.
- **`attn_implementation`** — attention compute is `O(L²)` while SDPA's memory-efficient kernel keeps *memory* `O(L)`. `sdpa` is the correct universal default; `flash_attention_2` is faster and unlocks multipack, but only on `sm_80`+.
- **Multi-node memory is declared, not coded.** For a 7 B+ CPT run the keys that matter are `deepspeed: deepspeed_configs/zero3_bf16.json` (shards optimiser + gradients + parameters across ranks — roughly `1/world_size` of the ~14·N bytes of optimiser state) or `fsdp_version: 2` with an `fsdp_config:` block. Nothing in the training loop changes.
- **Honest T4 throughput:** ~65 TFLOPS fp16 peak, realistically 25–35 % MFU with checkpointing → a 0.5 B model digests ~1 M tokens in 2–4 minutes, ~0.5 B tokens/day. Real knowledge injection wants 10⁹–10¹¹. ⚠️ **A T4 is a correctness rig for this pipeline, not a knowledge-injection rig** — which is precisely the argument for the declarative approach: the file you validate here is the file you scale.

#### Pros & Cons

**Pros**
- **The config is the experiment.** Version-controlled, diffable, reviewable in a PR, attachable to a model card. Reproducing a run six months later is `git checkout && axolotl train`.
- **Fail-fast schema validation** catches illegal *combinations* before any GPU time is spent.
- **Multi-node for free.** ZeRO-1/2/3, FSDP2, Ray, multi-machine `accelerate` — all declarative. The distributed boilerplate that eats a week is deleted, not written.
- **Tokenisation becomes a cacheable build step** shared across ranks and across runs.
- **The battle-tested defaults are the ones you would have had to discover** — LR schedule wiring, packing collators, `use_reentrant`, resume-from-checkpoint, safetensors sharding.
- **Artifacts are declarative too:** `save_steps` / `save_total_limit` / `hub_model_id` / `quantization:` (torchao PTQ) / `axolotl merge-lora` cover the full checkpoint lifecycle.

**Cons**
- **You inherit the framework's opinions, including the four above.** Two keys that read as synonyms (`type: completion` vs `type: pretrain`) produce measurably different training data. The abstraction does not remove the need to understand packing — it removes the need to *implement* it.
- **Debugging is one indirection deeper.** A stack trace lands inside Axolotl. Knowing which YAML key produced the frame is a skill you now need, on top of knowing PyTorch.
- **Version coupling is real.** Keys get deprecated (`flash_attention:` → `attn_implementation:`), defaults change, validators are added. **Pin the version in the same commit as the config,** or your reproducible file reproduces something else.
- **Anything genuinely novel fights the schema.** A custom loss, an unusual masking scheme, a bespoke curriculum — you write an Axolotl plugin or a custom prompt strategy, which is more work than the 30 lines it would have been in a script.
- **It does not curate your data**, which on a real CPT project is 80 % of the work and 100 % of the quality.
- **CPT's own trade-offs are untouched:** catastrophic forgetting without replay, LoRA underperforming full FT on knowledge acquisition specifically (*Biderman et al., 2024*), and an output that is a **base model** — a continuator, not an assistant.

#### Metrics to watch (there are no `rewards/*` keys in CPT)

- **`train_loss` / `eval_loss` on held-out *domain* text** — the primary signal, and here it is declarative: `test_datasets:` + `eval_steps:`. Should fall smoothly; jaggedness after warmup means the LR is too high for the re-warm.
- **Domain perplexity ↓ and general perplexity ↑ (the "forgetting tax")** — CPT is judged on the *pair*, never on one number. A few percent general regression is a healthy trade; 2× means the replay ratio or LR is wrong.
- **Bits-per-byte (BPB) instead of perplexity the moment you touch the vocabulary** (`tokens:` / `lora_modules_to_save: [embed_tokens, lm_head]`). Per-token PPL is not comparable across tokenisations; BPB normalises by UTF-8 bytes.
- **`grad_norm`** — a spike across the warmup boundary is expected; persistent spikes mean LR or data problems.
- **Token utilisation** (real tokens ÷ tokens forwarded) — measured directly off the prepared Arrow dataset in Step 4. Should be ~1.0; if it is not, `sample_packing` / `pretraining_sample_concatenation` is wrong and you are paying for padding.
- **The config hash / `dataset_prepared_path` directory name** — log it. It is the only thing that proves two runs saw identical tokens.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**CPT of `Qwen/Qwen2.5-0.5B` on clinical guidelines with 5 % general-text replay — expressed as one `config.yml` and driven by the `axolotl` CLI.**

> ⚙️ **What is deliberately *not* declarative here.** Two cells are Python on purpose:
> **(a) curation** — filtering `None` rows and sampling a replay slice is upstream of the training contract, and pretending otherwise would be dishonest about where the work lives;
> **(b) the before/after perplexity pair** — Axolotl's `test_datasets` gives you `eval_loss` on domain text declaratively, but the *forgetting tax* needs two separate held-out sets measured against the same model, which is an evaluation design decision, not a training one.
> Everything between those two — quantisation, adapter, tokenisation, chunking, batching, schedule, checkpointing — is YAML.

**Executable pipeline:**

| Step | What | Why it matters here |
|---|---|---|
| 0 | Install + **hardware probe** | `sm_75` → `sdpa`, `fp16`, no multipack. The probe *writes these into the config* — hardware-aware, not hardware-hardcoded |
| 1 | **Curation**: filter `epfl-llm/guidelines`, stream a `fineweb-edu` replay slice → 4 JSONL files | the one part the YAML cannot own |
| 2 | **Generate `cpt_qwen_t4.yml`** | the artifact. QLoRA + un-masked CLM + re-warm/re-decay + replay-as-a-list-entry |
| 3 | **Validate** against the pinned Pydantic schema | 200 ms, before any GPU time |
| 4 | `axolotl preprocess --debug` → inspect tokens, **measure packing utilisation** | proves the labels are un-masked and the chunking is right |
| 5 | Baseline perplexity, domain **and** general | CPT is judged on a before/after pair |
| 6 | `axolotl train` (single node) — and the **same file** under multi-node `accelerate` | the payoff |
| 7 | Artifacts: `merge-lora`, torchao PTQ `quantize`, hub push | declarative checkpoint lifecycle |
| 8 | After perplexity + **forgetting tax** table | the honest scorecard |

### Environment Setup

In [ ]:
# Axolotl pulls its own pinned torch/transformers/peft/trl/accelerate stack — let it.
# NOTE: flash-attn is deliberately NOT installed. It requires sm_80+ (Ampere); a Colab T4
# is sm_75, and `pip install flash-attn` there wastes ~20 min compiling something unusable.
# On A100/H100 use:  %pip install -q --no-build-isolation "axolotl[flash-attn,deepspeed]"
%pip install -q --no-build-isolation axolotl
%pip install -q "bitsandbytes>=0.43" "datasets>=3.0"

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

# ------------------------------------------------------------------ hardware probe
# Everything in this cell exists so that the YAML in Step 2 is GENERATED from the machine
# it will run on, instead of being copy-pasted with someone else's GPU baked into it.
HAS_CUDA = torch.cuda.is_available()
CC = torch.cuda.get_device_capability() if HAS_CUDA else (0, 0)
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "CPU"
SM = CC[0] * 10 + CC[1]

# bf16 needs Ampere (sm_80+). On Turing (T4, sm_75) fp16 is the only usable half precision,
# and tf32 does not exist at all -- setting it True there is a silent no-op at best.
SUPPORTS_BF16 = SM >= 80
# FlashAttention-2 kernels are compiled for sm_80+; its varlen path is what multipack rides on.
HAS_FLASH = importlib.util.find_spec("flash_attn") is not None and SUPPORTS_BF16

ATTN_IMPL = "flash_attention_2" if HAS_FLASH else "sdpa"
# Multipack (sample_packing) isolates packed documents with flash-attn varlen cu_seqlens.
# Without flash-attn it is either unavailable or (sdpa + bf16, pre-H100) a known 0.0-loss trap.
SAMPLE_PACKING = HAS_FLASH
BF16, FP16, TF32 = SUPPORTS_BF16, not SUPPORTS_BF16, SUPPORTS_BF16
COMPUTE_DTYPE = "bfloat16" if SUPPORTS_BF16 else "float16"

AX_VERSION = subprocess.run(
    [sys.executable, "-c", "import axolotl; print(axolotl.__version__)"],
    capture_output=True, text=True,
).stdout.strip() or "unknown"

print(f"axolotl: {AX_VERSION}")
print(f"torch: {torch.__version__}")
print(f"gpu: {GPU_NAME}  (sm_{SM})")
print("-" * 62)
print(f"{'derived YAML key':<28}{'value':>18}   reason")
print("-" * 62)
for k, v, why in [
    ("attn_implementation", ATTN_IMPL, "flash-attn needs sm_80+"),
    ("sample_packing", SAMPLE_PACKING, "multipack rides flash-attn varlen"),
    ("bf16", BF16, "bf16 needs sm_80+"),
    ("fp16", FP16, "Turing half-precision fallback"),
    ("tf32", TF32, "tf32 tensor cores are Ampere+"),
    ("bnb_4bit_compute_dtype", COMPUTE_DTYPE, "must match the training dtype"),
]:
    print(f"{k:<28}{str(v):>18}   # {why}")

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORKDIR)
(WORKDIR / "data").mkdir(exist_ok=True)
print(f"\nworkdir: {WORKDIR}")

### Step 1 — Curation: the part the YAML cannot own

Axolotl's data contract is *one column of text*. Getting a corpus **to** that contract is your job, and for these two datasets it is exactly three problems:

- **`epfl-llm/guidelines` contains `None` and stub rows** where a source was excluded for licensing. `type: completion` hands `field:` straight to the tokenizer, which raises on `None`. Filtering here is not optional.
- **`fineweb-edu` is TB-scale**, so the replay slice is **streamed** and capped. This is also why replay is a *ratio* decision made on token counts, not a `head -n`.
- **Held-out sets must be split before training**, and there must be **two** of them — domain and general — because CPT is scored on the pair. `test_datasets:` in the YAML will consume the domain one for `eval_loss`; both are used by the manual perplexity cells.

The output is four JSONL files with a single `text` column. That uniformity is the entire reason replay costs **one extra list entry** in the config instead of a code path.

> ⚠️ **What we deliberately do *not* do here: chunking.** Guideline documents run to tens of thousands of tokens. `type: completion` slices each document into as many `sequence_len`-token rows as it needs (up to `sequence_len × 64`), so pre-chunking would be re-implementing the framework. Note this is *not* true of `type: pretrain`, which truncates each document to `sequence_len - 2` before concatenating — sharp edge #3 from the Context Block.

In [ ]:
import random
from datasets import load_dataset
random.seed(42)
SE_LEN = 1024
DOMAIN_DOCS = 250    # long guideline documents -> Axolotl will slice these into many rows
REPLAY_RATIO = 0.05   # Ibrahim et al. 2024: 1-5% general-text replay
EVAL_DOMAIN  = 24
EVAL_GENERAL = 24
MIN_CHARS = 2_000  # drop stubs: a 200-char "see other guideline" row teaches nothing

# ---- Domain corpus -------------------------------------------------------------------
# Shuffle-then-slice a pool FIRST (cheap on Arrow), then filter inside the pool. Filtering
# all 38k long documents would rewrite ~800 MB of Arrow for no benefit.
try:
    raw = load_dataset("epfl-llm/guidelines", split="train")
    TEXT_COL = "clean_text"
except Exception as exc:  # gated / offline -> fall back to the other Section-1 corpus
    print(f"guidelines unavailable ({type(exc).__name__}); falling back to MedRAG/textbooks")
    raw = load_dataset("MedRAG/textbooks", split="train")
    TEXT_COL = "content"

pool = raw.shuffle(seed=42).select(range(min(4_000, len(raw))))
domain_texts = [
    t for t in pool[TEXT_COL]                       # None rows WILL appear here
    if isinstance(t, str) and len(t) >= MIN_CHARS
][: DOMAIN_DOCS + EVAL_DOMAIN]
assert len(domain_texts) > EVAL_DOMAIN, "domain pool too small — raise the pool size"

domain_eval, domain_train = domain_texts[:EVAL_DOMAIN], domain_texts[EVAL_DOMAIN:]
domain_chars = sum(len(t) for t in domain_train)

# ---- Replay corpus -------------------------------------------------------------------
# Streamed: fineweb-edu's 10BT sample is far larger than any Colab disk. We take documents
# until the replay slice reaches REPLAY_RATIO of the domain corpus BY CHARACTERS, which is
# the right unit -- "5% of documents" would be meaningless against 40k-token guidelines.
replay_target = int(REPLAY_RATIO * domain_chars)
replay_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True
).shuffle(seed=42, buffer_size=2_000)

replay_train, general_eval, acc = [], [], 0
for row in replay_stream:
    t = row["text"]
    if not isinstance(t, str) or len(t) < 500:
        continue
    if len(general_eval) < EVAL_GENERAL:
        general_eval.append(t)
    elif acc < replay_target:
        replay_train.append(t)
        acc += len(t)
    else:
        break

# ---- Write the contract: one `text` column, four files -------------------------------
def write_jsonl(path, texts):
    with open(path, "w", encoding="utf-8") as fh:
        for t in texts:
            fh.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
    return Path(path)

paths = {
    "domain":write_jsonl("data/cpt_domain.jsonl", domain_train),
    "replay":write_jsonl("data/cpt_replay.jsonl", replay_train),
    "eval_dom":write_jsonl("data/eval_domain.jsonl", domain_eval),
    "eval_gen":write_jsonl("data/eval_general.jsonl", general_eval),
}

print(f"{'file':<26}{'rows':>8}{'MB':>9}{'chars':>14}")
print("-" * 57)
for name, p in paths.items():
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p.name:<26}{n:>8}{p.stat().st_size/1e6:>9.2f}{'':>14}")
print(f"\nreplay is {100*acc/max(domain_chars,1):.2f}% of the domain corpus by character count")
print(f"~{(domain_chars+acc)/3.6/1e6:.2f}M tokens total (rough 3.6 chars/token estimate)")

### Step 2 — Generate `cpt_qwen_t4.yml`

**This cell is the notebook.** Everything before it is data prep; everything after it is a CLI call against this file.

Read it as the CPT recipe transcribed into a schema. Three things are worth noticing as you go:

- **Replay is a list entry** (`datasets[1]`), not a code path. Changing the mixture is a one-line diff — and on the finite `datasets:` path Axolotl merges *and shuffles* the entries (`shuffle_merged_datasets: true`), which is exactly what interleaved replay requires. This is the reason we are not on the streaming `pretraining_dataset:` path, where only element `[0]` is ever read.
- **The hardware keys come from the probe**, so this file is portable: the same generator on an H100 emits `flash_attention_2` / `bf16` / `sample_packing: true` and nothing else changes.
- **The CPT-specific choices are `train_on_inputs`, `lora_r`, `warmup_ratio`, and the base model** — the four values that make this continued pre-training rather than SFT.

In [ ]:
CONFIG_PATH = WORKDIR / "cpt_qwen_t4.yml"
OUTPUT_DIR = "./outputs/cpt-qwen-med"

config_yaml = f"""
# ==========================================================================================
# Continued Pre-training (CPT) / Domain Adaptation
#   Qwen2.5-0.5B (base)  x  clinical guidelines + 5% general-text replay
#
#   GENERATED by axolotl_declarative_tuning.ipynb
#   axolotl {AX_VERSION} | torch {torch.__version__} | {GPU_NAME} (sm_{SM})
#
#   THIS FILE IS THE EXPERIMENT. Commit it next to the pinned axolotl version; the run is
#   reproducible from these ~70 lines and the four JSONL files alone.
# ==========================================================================================

# ---- 1. Base checkpoint ------------------------------------------------------------------
# BASE, not -Instruct. CPT precedes instruction tuning; running an un-masked CLM loss over
# raw documents on an aligned checkpoint trains the model AWAY from its own chat template.
base_model: Qwen/Qwen2.5-0.5B
trust_remote_code: false
save_safetensors: true
seed: 42

# ---- 2. Quantisation + adapter (QLoRA) ---------------------------------------------------
# 4-bit weights are not differentiable, so the schema REQUIRES an adapter alongside
# load_in_4bit -- a cross-field validator, i.e. an error you get in 200ms instead of at step 0.
load_in_4bit: true
adapter: qlora
bnb_config_kwargs:
  bnb_4bit_quant_type: nf4              # NF4 is information-theoretically optimal for ~N(0,1) weights
  bnb_4bit_use_double_quant: true       # quantises the quantisation constants: ~0.4 bits/param saved
  bnb_4bit_compute_dtype: {COMPUTE_DTYPE:<18} # MUST match the training dtype below

# CPT needs CAPACITY, not style. Rank bounds how much new knowledge can be encoded, and
# Biderman et al. 2024 shows low rank hurts knowledge acquisition more than any other task.
# Compare: the DPO/ORPO notebooks in this repo use r=16 and are right to.
lora_r: 64
lora_alpha: 128                         # alpha = 2r, the standard pairing
lora_dropout: 0.05
lora_target_linear: true                # ALL linear projections incl. the MLP trio --
                                        # FFN blocks are where factual key-value memories live.
# ---- Vocabulary extension (the thing SFT structurally cannot do) --------------------------
# Uncomment BOTH to add domain tokens the base BPE shreds into 5-8 fragments. This trains the
# 151936 x 896 embedding matrix densely (~136M params, 4x the entire LoRA budget) -- it is the
# one option here that genuinely changes the VRAM picture, which is why it is off by default.
# tokens:
#   - hydroxychloroquine
#   - thrombocytopenia
#   - echocardiography
# lora_modules_to_save:
#   - embed_tokens
#   - lm_head

# ---- 3. Data contract --------------------------------------------------------------------
# type: completion  -> raw text from `field`, labels UNMASKED (no -100 anywhere), and each
# document is sliced into as many sequence_len rows as it needs. That is the CPT objective:
# 100% of tokens carry gradient, versus ~20-40% for completion-masked SFT.
datasets:
  - path: data/cpt_domain.jsonl
    ds_type: json
    split: train
    type: completion
    field: text
  # REPLAY. This single list entry is the entire anti-catastrophic-forgetting mechanism:
  # it makes the true objective (1-a)*L_domain + a*L_general. NOTE this only works on the
  # finite `datasets:` path -- `pretraining_dataset:` reads element [0] and silently drops
  # the rest, which would give you 0% replay and a lobotomised model.
  - path: data/cpt_replay.jsonl
    ds_type: json
    split: train
    type: completion
    field: text

shuffle_merged_datasets: true   # interleave domain and replay; sequential order is NOT replay
train_on_inputs: true           # CPT has no prompt/completion split -- nothing is masked
dataset_prepared_path: ./last_run_prepared   # hash-keyed Arrow cache: tokenise once, mmap on every rank
dataset_num_proc: 2

# Held-out DOMAIN text -> declarative eval_loss during training. The general-text half of the
# forgetting tax needs a second, separately-scored set, which is an evaluation-design decision
# and therefore lives in the notebook rather than here.
# `split: train` is correct even for a test set: a local JSONL only has a `train` split.
test_datasets:
  - path: data/eval_domain.jsonl
    ds_type: json
    split: train
    type: completion
    field: text
val_set_size: 0                 # mutually exclusive with test_datasets

# ---- 4. Sequence shaping -----------------------------------------------------------------
sequence_len: {SEQ_LEN}
pad_to_sequence_len: true       # static buffer shapes -> less allocator fragmentation, fewer late OOMs
sample_packing: {str(SAMPLE_PACKING).lower():<15} # multipack (BFD + block-diagonal mask); needs flash-attn varlen (sm_80+)
eval_sample_packing: false

# ---- 5. Hardware (derived from the probe, not hardcoded) ---------------------------------
attn_implementation: {ATTN_IMPL:<10} # sdpa is the correct universal default; O(L) memory, O(L^2) compute
bf16: {str(BF16).lower()}
fp16: {str(FP16).lower()}
tf32: {str(TF32).lower()}
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false          # non-optional with ZeRO-3 / frozen-parameter graphs like LoRA

# ---- 6. Optimisation: the Ibrahim et al. re-warm -> re-decay recipe -----------------------
num_epochs: 1                   # ONE pass. Repeats over raw text memorise verbatim; add tokens, not epochs.
micro_batch_size: 2
gradient_accumulation_steps: 8  # effective batch = 2 * 8 * {SEQ_LEN} = {2*8*SEQ_LEN:,} tokens/optimizer step
                                # NOTE: multiplied by world_size on multi-GPU. Scale accum DOWN when adding GPUs
                                # or the effective batch silently grows and the LR is no longer tuned for it.
learning_rate: 0.0002           # LoRA scale. Full-FT CPT would be 1e-5..3e-5.
                                # Written as a decimal on purpose: YAML 1.1 parses `2e-4` as a STRING.
lr_scheduler: cosine            # RE-DECAY into the domain basin
cosine_min_lr_ratio: 0.1
warmup_ratio: 0.05              # RE-WARM. A base checkpoint is the END of a decayed schedule and ships
                                # with no optimiser moments; without this the first batches poison them.
optimizer: paged_adamw_8bit     # 8-bit moments + paged states: ~0.35GB instead of ~5.9GB
weight_decay: 0.01
max_grad_norm: 1.0

# ---- 7. Artifacts & telemetry ------------------------------------------------------------
output_dir: {OUTPUT_DIR}
logging_steps: 5
eval_steps: 20
save_steps: 50
save_total_limit: 2
auto_resume_from_checkpoints: true   # preemptible/spot multi-node runs resume themselves
# save_first_step: true              # uncomment once to prove checkpoint saving works BEFORE a long run

special_tokens:
  pad_token: "<|endoftext|>"

# ---- 8. Experiment tracking (opt-in) -----------------------------------------------------
# wandb_project: cpt-domain-adaptation
# wandb_name: qwen05b-guidelines-r64
# wandb_mode: offline

# ---- 9. Publishing (opt-in) --------------------------------------------------------------
# hub_model_id: your-org/qwen2.5-0.5b-guidelines-cpt
# hub_strategy: checkpoint
""".lstrip()

CONFIG_PATH.write_text(config_yaml, encoding="utf-8")
print(f"wrote {CONFIG_PATH}  ({len(config_yaml.splitlines())} lines)")
print(config_yaml)

### Step 3 — Validate before you spend a GPU-minute

The single highest-leverage habit in declarative training: **parse the config against the schema of the exact Axolotl version you have pinned, before launching anything.**

- `validate_config` runs every cross-field `@model_validator` — the 4-bit-without-adapter check, the packing/attention checks, the `test_datasets` vs `val_set_size` exclusivity. A misconfiguration surfaces here in milliseconds instead of after the model has loaded.
- `axolotl config-schema --field <name>` prints the contract for a single key **in your installed version**, which is how you settle "was it `flash_attention:` or `attn_implementation:`?" without reading release notes. Key deprecations across versions are the most common cause of a config that used to work.

In [ ]:
import yaml

cfg_dict = yaml.safe_load(CONFIG_PATH.read_text())

# ---- 1. Full schema + cross-field validation -----------------------------------------
try:
    from axolotl.utils.config import validate_config
    from axolotl.utils.dict import DictDefault

    validated = validate_config(DictDefault(cfg_dict))
    print("schema validation: PASS")
    print(f"datasets declared: {len(cfg_dict['datasets'])}  (domain + replay)")
    print(f"adapter / bits: {cfg_dict['adapter']} / 4-bit={cfg_dict['load_in_4bit']}")
    print(f"attn / packing: {cfg_dict['attn_implementation']} / {cfg_dict['sample_packing']}")
except Exception as exc:
    # This is the branch that saves you 40 minutes. Read the message: Pydantic names the field.
    print(f"schema validation: FAIL\n  {type(exc).__name__}: {exc}")
    raise

# ---- 2. Interrogate the pinned schema for a specific key ------------------------------
# Deprecations are the #1 reason a working config stops working across upgrades.
for field in ("attn_implementation", "sample_packing", "pretraining_dataset"):
    out = subprocess.run(
        ["axolotl", "config-schema", "--field", field],
        capture_output=True, text=True,
    )
    body = (out.stdout or out.stderr).strip()
    print(f"\n--- {field} ---\n{body[:420]}")

### Step 4 — `axolotl preprocess`: tokenisation as a build artifact

`preprocess` runs the dataset half of the pipeline and nothing else — resolve → tokenise → apply the prompt strategy → (pack) → write Arrow to `dataset_prepared_path`, keyed by a hash of the dataset config.

Why this is a separate command and not a phase of training:

- It is **CPU-bound**. On a real CPT corpus it is tens of minutes during which every GPU in the job would sit idle.
- It is **cacheable**. Re-running with an unchanged data spec is instant; the hash changes only when the data spec does.
- It is **rank-safe**. One process writes; all ranks memory-map. Sixteen ranks independently tokenising the same 30 GB is the classic multi-node cold-start bug.

`--debug` prints tokenised samples with their labels. **This is where you confirm the CPT objective is actually what is being trained** — for `type: completion` every label must equal its `input_id`, with no `-100` anywhere. If you see masking, you are running SFT and calling it CPT.

In [ ]:
!axolotl preprocess {CONFIG_PATH.name} --debug 2>&1 | tail -n 45

In [ ]:
# ---- What the declarative packing actually bought us, in numbers ----------------------
# Read it straight off the prepared Arrow dataset rather than trusting the config.
from datasets import load_from_disk
from transformers import AutoTokenizer

prepared_root = WORKDIR / "last_run_prepared"
shards = sorted(p for p in prepared_root.iterdir() if p.is_dir()) if prepared_root.exists() else []
assert shards, "no prepared dataset — did preprocess fail?"

prepared = load_from_disk(str(shards[-1]))
lengths = [len(x) for x in prepared["input_ids"]]

real_tokens = sum(lengths)
forwarded = len(lengths) * SEQ_LEN   # pad_to_sequence_len => every row costs a full window
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

print(f"prepared rows: {len(lengths):,}")
print(f"real tokens: {real_tokens/1e6:.2f}M")
print(f"tokens forwarded: {forwarded/1e6:.2f}M")
print(f"TOKEN UTILISATION: {100*real_tokens/forwarded:.1f}%   <- ~100% is the goal")
print(f"optimizer steps (1 gpu): {len(lengths) // (2*8)}")

# ---- Prove the loss is un-masked ------------------------------------------------------
# CPT's defining property: labels == input_ids, everywhere. Any -100 means SFT-style masking
# leaked in and you are training on a fraction of the tokens you are paying to forward.
row = prepared[0]
n_masked = sum(1 for l in row["labels"] if l == -100)
print(f"\nrow 0: {len(row['input_ids'])} tokens, {n_masked} masked "
      f"({'UNMASKED — correct for CPT' if n_masked == 0 else 'MASKED — this is NOT CPT'})")
print("decoded head:", repr(tok.decode(row["input_ids"][:60])))

### Step 5 — Baseline perplexity, before a single gradient step

CPT has no reward and no accuracy, so the only honest report is a **before/after pair on two held-out sets**:

- **Domain perplexity** — did we learn the domain? (expect: large drop)
- **General perplexity** — what did we forget? (expect: small rise — the *forgetting tax*)

Reporting the first without the second is how people ship models that are fluent in cardiology and can no longer write an email. `test_datasets:` in the config gives Axolotl's `eval_loss` on the domain half for free during training; this cell adds the general half and the before-image.

We also measure **bits-per-byte**, which is tokenizer-invariant — mandatory the moment you enable the `tokens:` block, since per-token perplexity is not comparable across vocabularies.

In [ ]:
import gc
import math
import time

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if SUPPORTS_BF16 else torch.float16,
)

def make_blocks(jsonl_path, block=SEQ_LEN, max_blocks=40):
    """Concat-and-chunk the held-out text into fixed windows — the same shape training sees."""
    ids = []
    for line in open(jsonl_path, encoding="utf-8"):
        ids.extend(tok(json.loads(line)["text"], add_special_tokens=False)["input_ids"])
        ids.append(tok.eos_token_id)
        if len(ids) >= block * max_blocks:
            break
    return [ids[i:i + block] for i in range(0, len(ids) - block, block)][:max_blocks]

@torch.no_grad()
def eval_ppl_bpb(model, blocks, batch_size=2):
    """Token-level perplexity AND tokenizer-invariant bits-per-byte."""
    model.eval()
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in range(0, len(blocks), batch_size):
        chunk = blocks[i:i + batch_size]
        x = torch.tensor(chunk, device=model.device)
        # labels=input_ids -> the un-masked CLM objective, identical to what we train on.
        loss = model(input_ids=x, labels=x).loss.float()
        n_pred = x.numel() - len(chunk)        # HF shifts by one: N tokens -> N-1 predictions
        nll += loss.item() * n_pred
        n_tok += n_pred
        n_bytes += sum(len(tok.decode(c).encode("utf-8")) for c in chunk)
    return {"ppl": math.exp(nll / n_tok), "bpb": nll / n_bytes / math.log(2)}

domain_blocks  = make_blocks("data/eval_domain.jsonl")
general_blocks = make_blocks("data/eval_general.jsonl")
print(f"eval blocks: domain={len(domain_blocks)} general={len(general_blocks)}")

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", attn_implementation="sdpa"
)
before_domain = eval_ppl_bpb(base, domain_blocks)
before_general = eval_ppl_bpb(base, general_blocks)
print(f"BEFORE  domain  ppl={before_domain['ppl']:8.3f}  bpb={before_domain['bpb']:.4f}")
print(f"BEFORE  general ppl={before_general['ppl']:8.3f}  bpb={before_general['bpb']:.4f}")

# Free the baseline model — `axolotl train` runs in its own process and needs the whole GPU.
del base
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"\nGPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

### Step 6 — Train

One command. It reads the config, resolves the cached tokenised dataset, builds the quantised model plus adapter, wires the schedule, and launches under `accelerate`.

```bash
axolotl train cpt_qwen_t4.yml
```

`axolotl train` is a launcher wrapper — with `--launcher accelerate` (the default) it `exec`s the classic form, which is worth knowing because it is what you will see in cluster job scripts:

```bash
accelerate launch -m axolotl.cli.train cpt_qwen_t4.yml
```

Any config key can be overridden on the command line in kebab-case (`--learning-rate 0.0001`, `--lora-r 32`, `--micro-batch-size 1`) — useful for sweeps, **dangerous as a habit**: an override is not in the file, so it is not in `git`, and the run stops being reproducible. Sweep with `--sweep sweep.yml`, which materialises a real config per point.

**What to watch:** `loss` falling smoothly, `grad_norm` settling after the warmup boundary, and `eval_loss` on the held-out domain set dropping every `eval_steps: 20`. There are no reward metrics in CPT. A jagged curve after warmup means the LR is too high for the re-warm you configured.

In [ ]:
t0 = time.time()
!axolotl train {CONFIG_PATH.name}
print(f"\nwall clock: {(time.time() - t0)/60:.1f} min")

### Step 6b — The same file, on many machines

This is the payoff, and it is genuinely anticlimactic: **the config does not change.** Topology is supplied by the launcher and by two optional keys.

**Single node, N GPUs** — `accelerate` sees the local device count:
```bash
accelerate launch --num_processes 8 -m axolotl.cli.train cpt_qwen_t4.yml
```

**Two nodes × 8 GPUs** — identical command on both machines, one env var apart:
```bash
# on every node (rank 0 first):
accelerate launch \
  --num_machines 2 --num_processes 16 \
  --machine_rank $NODE_RANK \
  --main_process_ip $MASTER_ADDR --main_process_port 29500 \
  -m axolotl.cli.train cpt_qwen_t4.yml
```

**Sharding for models that do not fit** — one key, no code:
```yaml
deepspeed: deepspeed_configs/zero3_bf16.json   # `axolotl fetch deepspeed_configs`
# or FSDP2:
# fsdp_version: 2
# fsdp_config:
#   fsdp_offload_params: false
#   fsdp_state_dict_type: SHARDED_STATE_DICT
#   fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
#   fsdp_transformer_layer_cls_to_wrap: Qwen2DecoderLayer
```

Three things that bite in practice, none of which the YAML will warn you about:

1. **Run `axolotl preprocess` once, before the multi-node job**, on shared storage. Sixteen ranks racing to tokenise the same corpus is the classic cold-start failure.
2. **The effective batch scales with `world_size`.** `micro_batch_size × gradient_accumulation_steps × world_size × sequence_len`. Going 1 → 16 GPUs at fixed accumulation multiplies your batch by 16 and quietly invalidates the LR you tuned. Scale `gradient_accumulation_steps` **down**, or re-tune.
3. **`max_steps` on the streaming path counts optimizer steps, not tokens.** Sixteen ranks consume 16× the tokens per step, so the same `max_steps` is a 16× larger corpus pass.

In [ ]:
# Materialise the multi-node artifacts. Nothing here runs on a T4 — these are the files you
# commit alongside the config so the cluster run is `git clone && sbatch`, not tribal knowledge.

!axolotl fetch deepspeed_configs 2>&1 | tail -n 3

# ---- A scale-up config: 7B, streaming corpus, ZeRO-3, quantised export ------------------
# Note how few keys actually differ from cpt_qwen_t4.yml — and that NONE of them are code.
scale_yaml = f"""
base_model: Qwen/Qwen2.5-7B
trust_remote_code: false
save_safetensors: true
seed: 42

# ---- Streaming corpus: for 10^9+ tokens the dataset does not fit on disk ----------------
# type: pretrain tokenises on the fly. TWO sharp edges live in this block:
#  (1) ONLY element [0] is read -- a second entry for replay is silently ignored, so your
#      mixture MUST be pre-mixed into one source before it gets here.
#  (2) each document is truncated to sequence_len-2 BEFORE concatenation, so long-form
#      corpora lose their tails. Pre-chunk long documents during curation.
pretraining_dataset:
  - path: HuggingFaceFW/fineweb-edu     # <- replace with YOUR pre-mixed domain+replay corpus
    name: sample-10BT
    type: pretrain
    text_column: text
    split: train
pretraining_sample_concatenation: true  # NOT the default: without it every short doc is its own padded row
streaming_multipack_buffer_size: 10000
shuffle_merged_datasets: true

sequence_len: 4096
sample_packing: true
pretrain_multipack_attn: false          # raw-text CPT: cross-document attention is the pretraining
                                        # default and costs ~nothing. Set true for document isolation.
attn_implementation: flash_attention_2  # sm_80+ only; this is what multipack's varlen path needs

# Streaming datasets have no length, so the schema REQUIRES max_steps and FORBIDS val_set_size.
max_steps: 20000
val_set_size: 0

micro_batch_size: 1
gradient_accumulation_steps: 4          # effective batch = 1*4*world_size*4096 tokens
learning_rate: 0.00002                  # full-FT scale (1e-5..3e-5), NOT the 2e-4 used for LoRA
lr_scheduler: cosine
warmup_ratio: 0.02
optimizer: adamw_torch_fused
weight_decay: 0.01
max_grad_norm: 1.0

bf16: true
tf32: true
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false

# ---- Multi-node sharding: one key ------------------------------------------------------
# ZeRO-3 shards optimizer state + gradients + parameters across ranks (~1/world_size of the
# ~14*N bytes of AdamW state). The training loop is unchanged.
deepspeed: deepspeed_configs/zero3_bf16.json

output_dir: ./outputs/cpt-qwen7b-domain
logging_steps: 10
save_steps: 500
save_total_limit: 3
auto_resume_from_checkpoints: true      # spot/preemptible clusters resume themselves

# ---- Declarative quantised export: `axolotl quantize cpt_multinode.yml` -----------------
quantization:
  weight_dtype: int8                    # torchao PTQ; int4 | int8 | float8_e4m3fn | nvfp4 | mxfp4
  group_size: 32
  quantize_embedding: false

special_tokens:
  pad_token: "<|endoftext|>"
"""
(WORKDIR / "cpt_multinode.yml").write_text(scale_yaml.lstrip(), encoding="utf-8")

# ---- The launch script that goes in the repo next to it --------------------------------
launch_sh = """#!/usr/bin/env bash
# Identical on every node. Only NODE_RANK differs.
set -euo pipefail
: "${MASTER_ADDR:?set MASTER_ADDR to the rank-0 node IP}"
: "${NODE_RANK:?set NODE_RANK (0..NNODES-1)}"
NNODES=${NNODES:-2}
GPUS_PER_NODE=${GPUS_PER_NODE:-8}

# Tokenise ONCE, on rank 0, onto shared storage. Every other rank memory-maps the result.
if [ "$NODE_RANK" = "0" ]; then
  axolotl preprocess cpt_multinode.yml
fi

accelerate launch \\
  --num_machines "$NNODES" \\
  --num_processes $(( NNODES * GPUS_PER_NODE )) \\
  --machine_rank "$NODE_RANK" \\
  --main_process_ip "$MASTER_ADDR" \\
  --main_process_port 29500 \\
  -m axolotl.cli.train cpt_multinode.yml
"""
p = WORKDIR / "launch_multinode.sh"
p.write_text(launch_sh, encoding="utf-8")
p.chmod(0o755)
print("wrote cpt_multinode.yml and launch_multinode.sh")

### Step 7 — Artifact management

The checkpoint lifecycle is declarative too. What ran above already produced, per the config: adapter checkpoints every `save_steps: 50`, at most `save_total_limit: 2` retained, `eval_loss` logged every `eval_steps: 20`, and safetensors serialisation.

The three post-training operations, all CLI:

| Command | Produces | When you want it |
|---|---|---|
| `axolotl merge-lora <cfg>` | a standalone dense checkpoint with the adapter folded in | serving with vLLM/TGI, or **stacking SFT on top of CPT** — the usual next stage |
| `axolotl quantize <cfg>` | a torchao-quantised checkpoint per the `quantization:` block | deployment; `weight_dtype:` `int4` / `int8` / `float8_e4m3fn` / `nvfp4` / `mxfp4` |
| `hub_model_id:` + `hub_strategy:` | pushes checkpoints as they are saved | long cluster runs where local disk is ephemeral |

> ⚠️ **Merging into a 4-bit base is lossy.** `merge-lora` dequantises the NF4 base to bf16 before folding — the merged weights are not bit-identical to `base + adapter` at inference. For evaluation-grade fidelity, keep the adapter separate (as the perplexity cell below does) and merge only for serving.

In [ ]:
# Fold the adapter into a dense checkpoint. Skipped by default on a T4: a merged 0.5B model in
# bf16 is ~1GB of disk and the adapter alone is what the next pipeline stage (SFT) wants anyway.
MERGE = False

if MERGE:
    !axolotl merge-lora {CONFIG_PATH.name} --lora-model-dir {OUTPUT_DIR} 2>&1 | tail -n 12

adapter_dir = Path(OUTPUT_DIR)
print("\nartifacts:")
for p in sorted(adapter_dir.rglob("*"))[:25]:
    if p.is_file():
        print(f"  {p.relative_to(adapter_dir)!s:<48}{p.stat().st_size/1e6:>8.2f} MB")

# Training metrics Axolotl recorded, straight out of the HF trainer state.
state = adapter_dir / "trainer_state.json"
if state.exists():
    st = json.loads(state.read_text())
    hist = [h for h in st["log_history"] if "loss" in h or "eval_loss" in h]
    print(f"\n{'step':>6}{'loss':>12}{'eval_loss':>12}{'grad_norm':>12}{'lr':>12}")
    print("-" * 54)
    for h in hist[-12:]:
        print(f"{h.get('step',''):>6}{h.get('loss',''):>12}"
              f"{h.get('eval_loss',''):>12}{h.get('grad_norm',''):>12}"
              f"{h.get('learning_rate',''):>12}")

### Step 8 — After: domain gain vs. the forgetting tax

The same two held-out sets, the same metric, the base model with the adapter attached. Load the adapter **on top of** the 4-bit base rather than using a merged checkpoint — the comparison is only fair if the base half of the computation is bit-identical to the baseline measurement.

In [ ]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(model, OUTPUT_DIR)

after_domain  = eval_ppl_bpb(model, domain_blocks)
after_general = eval_ppl_bpb(model, general_blocks)

def pct(a, b):
    return 100 * (b - a) / a

print(f"{'set':<9}{'metric':<7}{'before':>11}{'after':>11}{'change':>11}   expected")
print("-" * 72)
for name, b, a, want in [
    ("domain",  before_domain,  after_domain,  "down strongly"),
    ("general", before_general, after_general, "up slightly (the tax)"),
]:
    for metric in ("ppl", "bpb"):
        print(f"{name:<9}{metric:<7}{b[metric]:>11.4f}{a[metric]:>11.4f}"
              f"{pct(b[metric], a[metric]):>10.2f}%   {want if metric=='ppl' else ''}")

tax = pct(before_general["ppl"], after_general["ppl"])
gain = -pct(before_domain["ppl"], after_domain["ppl"])
print(f"\ndomain gain {gain:.1f}%  |  forgetting tax {tax:+.1f}%")
print("a few % tax is a healthy trade; 2x means REPLAY_RATIO or the LR is wrong")

---

## **[Key Observations]**

*Fill in from the runs above. CPT is judged on a before/after pair, never on a single number — and a declarative run is judged additionally on whether the config alone reproduces it.*

### Run configuration

| Setting | Value |
|---|---|
| Axolotl version (pin it!) | |
| Base model | `Qwen/Qwen2.5-0.5B` (base) |
| `sequence_len` / `sample_packing` | |
| `attn_implementation` (derived from `sm_`) | |
| `DOMAIN_DOCS` / real domain tokens | |
| `REPLAY_RATIO` (actual %, by chars) | |
| LoRA `r` / `alpha` / `lora_target_linear` | |
| LR / `warmup_ratio` / scheduler | |
| Effective batch (tokens/optimizer step) | |
| `dataset_prepared_path` hash dir | |

### Results

| Metric | Before | After | Δ | Expected direction |
|---|---|---|---|---|
| **Domain** perplexity | | | | **↓ strongly** |
| **Domain** bits-per-byte | | | | ↓ |
| **General** perplexity | | | | ↑ *slightly* (forgetting tax) |
| **General** bits-per-byte | | | | ↑ slightly |
| Final `train_loss` | | | | — |
| Final `eval_loss` (domain, declarative) | | | | ↓ |

### Efficiency & hardware

| Metric | Value |
|---|---|
| Token utilisation from the prepared dataset (%) | |
| Prepared rows / optimizer steps | |
| Peak VRAM (GB) | |
| Wall clock (min) | |
| Trainable params (% of total) | |

### Declarative-specific

- Did **schema validation** catch anything before launch? Which field?
- Did `preprocess --debug` show **zero `-100` labels**? (If not, it is not CPT.)
- What is the token utilisation, and would `sample_packing: true` on an Ampere GPU have changed it? By how much, given your row-length distribution?
- **Diff `cpt_qwen_t4.yml` against `cpt_multinode.yml`**: how many of the differing keys are *algorithmic* (LR, schedule, corpus) versus *topological* (deepspeed, attention, batch shape)? That ratio is the argument for this whole approach.

### Things worth logging every time

- The **exact Axolotl version** next to the config. A config without a version pin is not reproducible.
- `grad_norm` across the warmup boundary (spike → settle is healthy).
- Sensitivity sweep worth running: `REPLAY_RATIO ∈ {0, 0.01, 0.05, 0.15}` against the forgetting tax — with `--sweep`, so each point materialises a real config file. `0` is the instructive control.
- Whether `axolotl preprocess` was run **once** ahead of a multi-rank job, or 16 times by accident.

## Export — Download the Domain-Adapted Adapter (Optional)

> Ship the **config next to the weights.** An adapter without the YAML that produced it is an artifact nobody can reproduce, retrain, or audit — so the archive below contains both.

In [ ]:
import shutil

bundle = WORKDIR / "cpt_axolotl_bundle"
bundle.mkdir(exist_ok=True)
if Path(OUTPUT_DIR).exists():
    shutil.copytree(OUTPUT_DIR, bundle / "adapter", dirs_exist_ok=True)
shutil.copy(CONFIG_PATH, bundle / CONFIG_PATH.name)          # the experiment
shutil.copy(WORKDIR / "cpt_multinode.yml", bundle / "cpt_multinode.yml")
shutil.copy(WORKDIR / "launch_multinode.sh", bundle / "launch_multinode.sh")
(bundle / "VERSIONS.txt").write_text(f"axolotl=={AX_VERSION}\ntorch=={torch.__version__}\n")

output_filename = "cpt_axolotl_bundle.zip"
shutil.make_archive(output_filename.replace(".zip", ""), "zip", bundle)
print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found — run the export cell first.")

---

## Model Usage — **Continuation**, not chat

A CPT'd model is still a **base model**: a text continuator. There is no chat template, no `add_generation_prompt`, and asking it a question gets you a plausible *continuation of the question*, not an answer. That is not a bug — instruction tuning is the **next** stage, and it consumes this checkpoint as its starting point.

Axolotl has a declarative inference path too — `axolotl inference cpt_qwen_t4.yml --lora-model-dir ./outputs/cpt-qwen-med` (add `--gradio` for a UI) — but the A/B below is more informative: `PeftModel.disable_adapter()` gives the *same* quantised base model with the adapter switched off, so any difference in the continuation is attributable to the CPT run and nothing else.

In [ ]:
@torch.no_grad()
def continue_text(prefix, max_new_tokens=90, use_adapter=True):
    """Raw continuation — NO chat template. A CPT'd base model completes text, it does not answer."""
    inputs = tok(prefix, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    def _gen():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if use_adapter:
        return _gen()
    with model.disable_adapter():   # exact same 4-bit base weights, adapter off
        return _gen()

PROMPTS = [
    "In patients with suspected acute coronary syndrome, the recommended initial",
    "First-line pharmacological management of type 2 diabetes mellitus consists of",
]

for p in PROMPTS:
    print("=" * 88)
    print(f"PREFIX: {p}\n")
    print(f"[BASE     ] {continue_text(p, use_adapter=False)}\n")
    print(f"[CPT'd    ] {continue_text(p, use_adapter=True)}")

print("\nLook for domain REGISTER — guideline phrasing, evidence grades, drug names, dosing —")
print("not factual correctness. On 250 documents you are buying register; knowledge needs 10^9 tokens.")